In [ ]:
import os
open('/kaggle/working/viz_l3.py','w',encoding='utf-8').write('"""Visualize REAL nuScenes-mini val samples: 6 cameras + 3D pred/GT boxes + LiDAR BEV, for two official models.\nNo nuscenes-devkit needed: reads the json tables directly. Usage: see argparse below."""\nimport argparse, csv, json, os, sys, zipfile\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.gridspec import GridSpec\nfrom PIL import Image\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--root\', required=True, help=\'nuScenes-mini root (has samples/ sweeps/ v1.0-mini/)\')\nap.add_argument(\'--tables\', default=None, help=\'dir with json tables (default: <root>/v1.0-mini)\')\nap.add_argument(\'--res_lidar\', required=True)\nap.add_argument(\'--res_cam\', required=True)\nap.add_argument(\'--out\', required=True)\nap.add_argument(\'--thr\', type=float, default=0.3)\nap.add_argument(\'--limit\', type=int, default=0)\nap.add_argument(\'--mock\', action=\'store_true\', help=\'local code-path test: fake images/points if files are missing\')\nap.add_argument(\'--dpi\', type=int, default=75)\nargs = ap.parse_args()\nTAB = args.tables or os.path.join(args.root, \'v1.0-mini\')\nos.makedirs(args.out, exist_ok=True)\n\n\ndef load_table(name):\n    p = os.path.join(TAB, name + \'.json\')\n    if os.path.exists(p):\n        return json.load(open(p, encoding=\'utf-8\'))\n    with zipfile.ZipFile(p + \'.zip\') as z:\n        return json.load(z.open(z.namelist()[0]))\n\n\ndef qR(q):\n    w, x, y, z = q\n    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],\n                     [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],\n                     [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])\n\n\nsample = {s[\'token\']: s for s in load_table(\'sample\')}\nsd_all = load_table(\'sample_data\')\nego = {e[\'token\']: e for e in load_table(\'ego_pose\')}\ncs = {c[\'token\']: c for c in load_table(\'calibrated_sensor\')}\nscene = {s[\'token\']: s for s in load_table(\'scene\')}\ninst = {i[\'token\']: i for i in load_table(\'instance\')}\ncat = {c[\'token\']: c[\'name\'] for c in load_table(\'category\')}\nkf = {}\nfor d in sd_all:\n    if d[\'is_key_frame\']:\n        kf[(d[\'sample_token\'], d[\'filename\'].split(\'/\')[1])] = d\nann = {}\nfor a in load_table(\'sample_annotation\'):\n    ann.setdefault(a[\'sample_token\'], []).append(a)\n\nGEN2DET = {\'vehicle.car\': \'car\', \'vehicle.truck\': \'truck\', \'vehicle.bus.bendy\': \'bus\', \'vehicle.bus.rigid\': \'bus\',\n           \'vehicle.trailer\': \'trailer\', \'vehicle.construction\': \'construction_vehicle\',\n           \'vehicle.motorcycle\': \'motorcycle\', \'vehicle.bicycle\': \'bicycle\',\n           \'human.pedestrian.adult\': \'pedestrian\', \'human.pedestrian.child\': \'pedestrian\',\n           \'human.pedestrian.construction_worker\': \'pedestrian\', \'human.pedestrian.police_officer\': \'pedestrian\',\n           \'movable_object.trafficcone\': \'traffic_cone\', \'movable_object.barrier\': \'barrier\'}\n\nR_l = json.load(open(args.res_lidar))[\'results\']\nR_c = json.load(open(args.res_cam))[\'results\']\ntokens = sorted(R_l.keys(), key=lambda t: (sample[t][\'scene_token\'], sample[t][\'timestamp\']))\nassert set(R_l) == set(R_c), \'two result files cover different samples\'\nif args.limit:\n    tokens = tokens[:args.limit]\n\nXC = np.array([1, 1, 1, 1, -1, -1, -1, -1.])\nYC = np.array([1, -1, -1, 1, 1, -1, -1, 1.])\nZC = np.array([1, 1, -1, -1, 1, 1, -1, -1.])\nEDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]\n\n\ndef corners(center, size, quat):\n    w, l, h = size\n    c = np.stack([XC * l / 2, YC * w / 2, ZC * h / 2])\n    return (qR(quat) @ c).T + np.array(center)\n\n\ndef gt_boxes(t):\n    out = []\n    for a in ann.get(t, []):\n        n = cat[inst[a[\'instance_token\']][\'category_token\']]\n        if n in GEN2DET:\n            out.append(dict(translation=a[\'translation\'], size=a[\'size\'], rotation=a[\'rotation\'],\n                            detection_name=GEN2DET[n], detection_score=1.0))\n    return out\n\n\ndef to_ego(P, e):\n    return (qR(e[\'rotation\']).T @ (P - np.array(e[\'translation\'])).T).T\n\n\ndef to_cam(Pe, c):\n    return (qR(c[\'rotation\']).T @ (Pe - np.array(c[\'translation\'])).T).T\n\n\ndef draw_cam(ax, img, boxes, sd, color, lw, label_scores):\n    c = cs[sd[\'calibrated_sensor_token\']]\n    e = ego[sd[\'ego_pose_token\']]\n    K = np.array(c[\'camera_intrinsic\'])\n    H, W = img.shape[:2]\n    s = W / 1600.0\n    if not ax.images:\n        ax.imshow(img)\n    n_vis = 0\n    for b in boxes:\n        P = corners(b[\'translation\'], b[\'size\'], b[\'rotation\'])\n        Pc = to_cam(to_ego(P, e), c)\n        if np.all(Pc[:, 2] < 0.1):\n            continue\n        uv = (K @ Pc.T).T\n        z = np.maximum(uv[:, 2:3], 1e-3)\n        uv = uv[:, :2] / z * s\n        drawn = False\n        for i, j in EDGES:\n            if Pc[i, 2] > 0.1 and Pc[j, 2] > 0.1:\n                ax.plot([uv[i, 0], uv[j, 0]], [uv[i, 1], uv[j, 1]], color=color, lw=lw)\n                drawn = True\n        if drawn:\n            n_vis += 1\n            if label_scores:\n                ok = np.where(Pc[:, 2] > 0.1)[0]\n                p = uv[ok].mean(0)\n                if 0 <= p[0] < W and 0 <= p[1] < H:\n                    ax.text(p[0], p[1], \'%s %.2f\' % (b[\'detection_name\'][:4], b[\'detection_score\']), color=\'yellow\',\n                            fontsize=5, ha=\'center\', va=\'center\')\n    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.axis(\'off\')\n    return n_vis\n\n\ndef draw_bev(ax, pts, boxes_pred, boxes_gt, e, title):\n    ax.set_facecolor(\'black\')\n    ax.scatter(-pts[:, 1], pts[:, 0], s=0.25, c=np.clip(pts[:, 2], -3, 3), cmap=\'Greys_r\', vmin=-3, vmax=3, linewidths=0)\n    for boxes, col, lw in ((boxes_gt, \'#39ff14\', 1.0), (boxes_pred, \'#ff3030\', 1.1)):\n        for b in boxes:\n            P = to_ego(corners(b[\'translation\'], b[\'size\'], b[\'rotation\']), e)\n            poly = P[[0, 1, 5, 4, 0]]\n            ax.plot(-poly[:, 1], poly[:, 0], color=col, lw=lw)\n            f = (P[0] + P[1]) / 2\n            ax.plot([-(P[:4, 1].mean()), -f[1]], [P[:4, 0].mean(), f[0]], color=col, lw=lw)\n    ax.plot(0, 0, marker=\'^\', color=\'cyan\', ms=6)\n    ax.set_xlim(-54, 54); ax.set_ylim(-54, 54); ax.set_aspect(\'equal\')\n    ax.tick_params(labelsize=6, colors=\'gray\')\n    ax.set_title(title, fontsize=8)\n\n\nCAMS = [[\'CAM_FRONT_LEFT\', \'CAM_FRONT\', \'CAM_FRONT_RIGHT\'], [\'CAM_BACK_LEFT\', \'CAM_BACK\', \'CAM_BACK_RIGHT\']]\nstats = []\nfor idx, t in enumerate(tokens, 1):\n    s = sample[t]\n    sc = scene[s[\'scene_token\']]\n    lid = kf[(t, \'LIDAR_TOP\')]\n    e_l = ego[lid[\'ego_pose_token\']]\n    c_l = cs[lid[\'calibrated_sensor_token\']]\n    fp = os.path.join(args.root, lid[\'filename\'])\n    if os.path.exists(fp):\n        pts = np.fromfile(fp, dtype=np.float32).reshape(-1, 5)[:, :3]\n    elif args.mock:\n        pts = np.random.randn(20000, 3) * [20, 20, 1]\n    else:\n        raise FileNotFoundError(fp)\n    pts_e = (qR(c_l[\'rotation\']) @ pts.T).T + np.array(c_l[\'translation\'])\n    pl = [b for b in R_l[t] if b[\'detection_score\'] >= args.thr]\n    pc = [b for b in R_c[t] if b[\'detection_score\'] >= args.thr]\n    gt = gt_boxes(t)\n\n    fig = plt.figure(figsize=(24, 9.6))\n    gs = GridSpec(2, 5, figure=fig, width_ratios=[1, 1, 1, 1.05, 1.05], wspace=0.04, hspace=0.06)\n    n_img, n_pred_vis, cam_ok = 0, 0, 0\n    for r in range(2):\n        for k in range(3):\n            ch = CAMS[r][k]\n            d = kf[(t, ch)]\n            ip = os.path.join(args.root, d[\'filename\'])\n            ax = fig.add_subplot(gs[r, k])\n            if os.path.exists(ip):\n                im = Image.open(ip).convert(\'RGB\')\n                im = im.resize((640, 360))\n                img = np.asarray(im); cam_ok += 1\n            elif args.mock:\n                img = np.full((360, 640, 3), 90, np.uint8)\n            else:\n                raise FileNotFoundError(ip)\n            draw_cam(ax, img, gt, d, \'#39ff14\', 0.6, False)\n            n_pred_vis += draw_cam(ax, img, pc, d, \'#ff3030\', 1.0, True)\n            ax.set_title(ch, fontsize=8, color=\'black\', pad=2)\n    ax_l = fig.add_subplot(gs[:, 3])\n    draw_bev(ax_l, pts_e, pl, gt, e_l, \'L3C official LiDAR-only | %d pred (score>=%.2f) vs %d GT\' % (len(pl), args.thr, len(gt)))\n    ax_c = fig.add_subplot(gs[:, 4])\n    draw_bev(ax_c, pts_e, pc, gt, e_l, \'L3D official LiDAR+Camera | %d pred (score>=%.2f) vs %d GT\' % (len(pc), args.thr, len(gt)))\n    fig.suptitle(\'sample %d/%d | %s (%s) | token %s | t=%.3fs | LiDAR pts %d\\n\'\n                 \'cameras: GT green, L3D pred red (yellow = class score) | BEV: GT green, pred red | official pretrained BEVFusion on nuScenes-mini VAL (contaminated, not a benchmark)\'\n                 % (idx, len(tokens), sc[\'name\'], sc[\'description\'][:38], t[:8], s[\'timestamp\'] / 1e6, len(pts)), fontsize=9)\n    out = os.path.join(args.out, \'sample_%03d_%s.png\' % (idx, sc[\'name\']))\n    fig.savefig(out, dpi=args.dpi, bbox_inches=\'tight\')\n    plt.close(fig)\n    stats.append(dict(idx=idx, token=t, scene=sc[\'name\'], cams_found=cam_ok, lidar_pts=int(len(pts)), gt=len(gt),\n                      pred_lidar=len(pl), pred_cam=len(pc), file=os.path.basename(out)))\n    if idx % 10 == 0 or idx == len(tokens):\n        print(\'rendered\', idx, \'/\', len(tokens), flush=True)\n\nwith open(os.path.join(args.out, \'viz_index.csv\'), \'w\', newline=\'\') as f:\n    w = csv.DictWriter(f, fieldnames=list(stats[0].keys()))\n    w.writeheader(); w.writerows(stats)\nsummary = dict(n_samples=len(stats), thr=args.thr, cams_missing=sum(6 - x[\'cams_found\'] for x in stats),\n               mean_gt=float(np.mean([x[\'gt\'] for x in stats])), mean_pred_lidar=float(np.mean([x[\'pred_lidar\'] for x in stats])),\n               mean_pred_cam=float(np.mean([x[\'pred_cam\'] for x in stats])), mock=args.mock)\njson.dump(summary, open(os.path.join(args.out, \'viz_summary.json\'), \'w\'), indent=2)\npick = [stats[i][\'file\'] for i in np.linspace(0, len(stats) - 1, min(6, len(stats))).astype(int)]\nims = [Image.open(os.path.join(args.out, f)) for f in pick]\nWm = 1200\nims = [im.resize((Wm, int(im.height * Wm / im.width))) for im in ims]\nsheet = Image.new(\'RGB\', (2 * Wm, sum(im.height for im in ims[0::2]) + 0), \'white\')\ny = [0, 0]\nfor i, im in enumerate(ims):\n    sheet.paste(im, ((i % 2) * Wm, y[i % 2])); y[i % 2] += im.height\nsheet.save(os.path.join(args.out, \'overview_6_samples.png\'))\nprint(json.dumps(summary, indent=2))\nok = summary[\'n_samples\'] == (args.limit or 81) and (args.mock or summary[\'cams_missing\'] == 0)\nprint(\'VIZ PASS\' if ok else \'VIZ FAIL\')\n')
print('written', os.path.getsize('/kaggle/working/viz_l3.py'))

In [ ]:
%%bash
set -euo pipefail
V=$(find /kaggle/input -maxdepth 6 -type d -name v1.0-mini | head -1); ROOT=$(dirname "$V")
RL=$(find /kaggle/input -name results_nusc.json -path '*L3C_lidar_only*' | head -1)
RC=$(find /kaggle/input -name results_nusc.json -path '*L3D_lidar_camera*' | head -1)
echo "ROOT=$ROOT"; echo "RL=$RL"; echo "RC=$RC"
[ -n "$ROOT" ] && [ -n "$RL" ] && [ -n "$RC" ] || { find /kaggle/input -maxdepth 4 | head -50; exit 1; }
mkdir -p /kaggle/working/viz
python3 /kaggle/working/viz_l3.py --root "$ROOT" --res_lidar "$RL" --res_cam "$RC" --out /kaggle/working/viz --thr 0.3 2>&1 | tail -30
ls /kaggle/working/viz | wc -l
